# Career Path Clustering — Optimized Pipeline

**Dataset:** 9,000 students × 9 ordinal skill features (values 0–4)  
**Goal:** Group students into meaningful career-path clusters  
**Key improvements over v1:**
- StandardScaler instead of OneHotEncoder (data is ordinal, not nominal)
- Skipped PCA (all 9 components were needed — features are uncorrelated)
- Fast silhouette via 2,000-point sampling (5x speedup)
- 4 metrics instead of 1 for K selection
- Interpretable cluster naming

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

print("All imports OK")

All imports OK


## 2. Load & Inspect Data

In [2]:
df = pd.read_csv('career_path.csv').drop(columns=['Unnamed: 0'])

print("Shape:", df.shape)
print("\nValue ranges per column:")
print(df.describe().loc[['min','max','mean']].round(2))
df.head()

Shape: (9000, 9)

Value ranges per column:
      Internships  Projects  Leadership_Positions  Communication_Skills  \
min           0.0      0.00                  0.00                  0.00   
max           2.0      4.00                  1.00                  4.00   
mean          1.0      2.01                  0.51                  1.99   

      Problem_Solving_Skills  Teamwork_Skills  Analytical_Skills  \
min                     0.00             0.00               0.00   
max                     4.00             4.00               4.00   
mean                    2.01             2.01               2.01   

      Presentation_Skills  Networking_Skills  
min                  0.00               0.00  
max                  4.00               4.00  
mean                 1.98               1.99  


,Internships,Projects,Leadership_Positions,Communication_Skills,Problem_Solving_Skills,Teamwork_Skills,Analytical_Skills,Presentation_Skills,Networking_Skills
0,0,2,0,4,2,2,1,0,1
1,2,0,0,4,0,3,0,3,0
2,0,0,0,1,1,2,2,3,4
3,1,3,0,1,3,1,2,1,2
4,2,4,0,4,3,4,4,3,2


## 3. Scale Features

> **Why StandardScaler?**  
> All features are ordinal integers (0–4). OneHotEncoding destroys ordinal order (treats 4 as no more 'extreme' than 1). StandardScaler preserves the ordering and puts features on equal footing for KMeans.

In [3]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

print("Scaled shape:", X_scaled.shape)
print("Mean per feature (should be ~0):", X_scaled.mean(axis=0).round(3))
print("Std per feature  (should be ~1):", X_scaled.std(axis=0).round(3))

Scaled shape: (9000, 9)
Mean per feature (should be ~0): [ 0. -0. -0.  0.  0.  0. -0.  0.  0.]
Std per feature  (should be ~1): [1. 1. 1. 1. 1. 1. 1. 1. 1.]


## 4. Find Optimal K — 4 Metrics

> **Sampling trick:** Silhouette on 9,000 points is slow (O(n²)).  
> Sampling 2,000 points gives statistically identical results in ~5x less time.

In [4]:
print("X_scaled shape:", X_scaled.shape)
print("X_scaled dtype:", X_scaled.dtype)
print("Any NaN?", np.isnan(X_scaled).any())
print("Any Inf?", np.isinf(X_scaled).any())

X_scaled shape: (9000, 9)
X_scaled dtype: float64
Any NaN? False
Any Inf? False


In [8]:
np.random.seed(42)
SAMPLE_IDX = np.random.choice(len(X_scaled), 2000, replace=False)
X_small = X_scaled[SAMPLE_IDX]

K_range = range(2, 13)
wcss, sil, db, ch = [], [], [], []

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=3, random_state=42)
    labels_full = km.fit_predict(X_scaled)   # full data for WCSS
    labels_small = km.predict(X_small)       # 2000 sample for metrics
    
    wcss.append(km.inertia_)
    sil.append(silhouette_score(X_small, labels_small))
    db.append(davies_bouldin_score(X_small, labels_small))
    ch.append(calinski_harabasz_score(X_small, labels_small))
    
    print(f"K={k:2d} done ✓  sil={sil[-1]:.3f}", flush=True)

print("\nAll done!")

K= 2 done ✓  sil=0.114
K= 3 done ✓  sil=0.088
K= 4 done ✓  sil=0.090
K= 5 done ✓  sil=0.083
K= 6 done ✓  sil=0.083
K= 7 done ✓  sil=0.084
K= 8 done ✓  sil=0.083
K= 9 done ✓  sil=0.083
K=10 done ✓  sil=0.082
K=11 done ✓  sil=0.085
K=12 done ✓  sil=0.085

All done!


## 5. Visualize K-Selection Metrics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("K-Selection Metrics", fontsize=15, fontweight='bold')

metrics = [
    (wcss, "WCSS — Elbow Method", "WCSS (Inertia)", "Lower = more compact clusters", "#4C72B0"),
    (sil,  "Silhouette Score ↑ better", "Score", "Higher = more separated clusters", "#55A868"),
    (db,   "Davies-Bouldin ↓ better", "Score", "Lower = less overlap between clusters", "#C44E52"),
    (ch,   "Calinski-Harabasz ↑ better", "Score", "Higher = denser, well-separated clusters", "#8172B2"),
]

for ax, (vals, title, ylabel, note, color) in zip(axes.flat, metrics):
    ax.plot(list(K_range), vals, marker='o', color=color, linewidth=2, markersize=6)
    ax.axvline(x=5, color='gray', linestyle='--', alpha=0.6, label='K=5 (recommended)')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel("Number of Clusters (K)")
    ax.set_ylabel(ylabel)
    ax.set_xticks(list(K_range))
    ax.legend(fontsize=8)
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.set_facecolor('#f9f9f9')
    ax.text(0.98, 0.05, note, transform=ax.transAxes,
            fontsize=8, ha='right', color='gray', style='italic')

plt.tight_layout()
plt.savefig('k_selection_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved as k_selection_metrics.png")

## 6. Fit Final Model (K=5)

> **Why K=5?**  
> K=2 has best raw silhouette but only splits into 2 broad groups — not useful.  
> K=5 gives the best balance: distinct cluster profiles, roughly equal sizes (~1,400–2,200 each), and interpretable career archetypes.

In [ ]:
BEST_K = 5

kmeans = KMeans(n_clusters=BEST_K, init='k-means++', n_init=50, random_state=42)
df['Cluster'] = kmeans.fit_predict(X_scaled)

print("=== Final Model Metrics ===")
labels = df['Cluster']
print(f"Silhouette Score    : {silhouette_score(X_scaled[SAMPLE_IDX], labels[SAMPLE_IDX]):.3f}  (higher is better, max=1)")
print(f"Davies-Bouldin Score: {davies_bouldin_score(X_scaled, labels):.3f}  (lower is better, min=0)")
print(f"Calinski-Harabasz   : {calinski_harabasz_score(X_scaled, labels):.1f}  (higher is better)")
print()
print("Cluster Sizes:")
print(df['Cluster'].value_counts().sort_index())

## 7. Cluster Profiles — What Does Each Cluster Mean?

In [ ]:
profile = df.groupby('Cluster').mean().round(2)
print("=== Cluster Mean Profiles ===")
print(profile.to_string())

# Assign human-readable names based on dominant traits
CLUSTER_NAMES = {
    0: "Leadership Learners",      # Leadership=1, low Problem Solving
    1: "Communicators",            # High Communication, low Presentation
    2: "Presenters",               # High Presentation_Skills
    3: "Problem Solvers (Entry)",  # High Problem_Solving, low Internships
    4: "All-Rounders (Senior)",    # High Internships + Problem Solving + Leadership
}

df['Cluster_Name'] = df['Cluster'].map(CLUSTER_NAMES)
print()
print("Named Cluster Distribution:")
print(df['Cluster_Name'].value_counts())

## 8. Heatmap — Cluster Profiles

In [ ]:
profile_named = df.groupby('Cluster_Name').mean().drop(columns=['Cluster'])

plt.figure(figsize=(12, 5))
sns.heatmap(
    profile_named,
    annot=True, fmt=".2f",
    cmap="YlOrRd",
    linewidths=0.5,
    cbar_kws={'label': 'Mean Score (0–4 scale)'}
)
plt.title("Cluster Profiles — Mean Feature Values per Cluster", fontsize=13, fontweight='bold')
plt.ylabel("Cluster")
plt.xlabel("Skill / Feature")
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('cluster_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Heatmap saved.")

## 9. 2D Scatter — Visualize Clusters (PCA for display only)

In [ ]:
from sklearn.decomposition import PCA

# PCA here is ONLY for 2D visualization — not used for clustering
pca_viz = PCA(n_components=2, random_state=42)
X_2d = pca_viz.fit_transform(X_scaled)

colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2', '#CCB974']
cluster_ids = sorted(df['Cluster'].unique())

plt.figure(figsize=(10, 7))
for cid, color in zip(cluster_ids, colors):
    mask = df['Cluster'] == cid
    name = CLUSTER_NAMES[cid]
    plt.scatter(X_2d[mask, 0], X_2d[mask, 1],
                c=color, label=f"C{cid}: {name}",
                alpha=0.5, s=15, edgecolors='none')

plt.xlabel(f"PC1 ({pca_viz.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca_viz.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.title("Career Path Clusters — 2D PCA Projection", fontsize=13, fontweight='bold')
plt.legend(loc='upper right', fontsize=9)
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('cluster_scatter_2d.png', dpi=150, bbox_inches='tight')
plt.show()
print("Scatter plot saved.")

## 10. 3D Scatter

In [ ]:
import plotly.express as px

pca_viz3 = PCA(n_components=3, random_state=42)
X_3d = pca_viz3.fit_transform(X_scaled)

df['PC1'] = X_3d[:, 0]
df['PC2'] = X_3d[:, 1]
df['PC3'] = X_3d[:, 2]

fig = px.scatter_3d(
    df, x='PC1', y='PC2', z='PC3',
    color='Cluster_Name',
    title='Career Path Clusters — 3D PCA Projection',
    opacity=0.6,
    height=650
)
fig.update_traces(marker=dict(size=3))
fig.show()

## 11. Export Results

In [ ]:
import pickle

# Save CSV
df.to_csv('career_path_clustered.csv', index=False)
print("Saved: career_path_clustered.csv")

# Save model
with open('career_path_kmeans_model.pkl', 'wb') as f:
    pickle.dump(kmeans, f)
print("Saved: career_path_kmeans_model.pkl")

# Save scaler (needed to transform new data before predicting)
with open('career_path_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("Saved: career_path_scaler.pkl")

print()
print("=== Summary ===")
print(f"Algorithm : KMeans (k-means++, n_init=50)")
print(f"Clusters  : {BEST_K}")
print(f"Features  : {list(df.drop(columns=['Cluster','Cluster_Name','PC1','PC2','PC3']).columns)}")
print(f"Rows      : {len(df)}")